# 04 - Statistical validation

Segment boundaries drawn by an unsupervised method are a hypothesis. This
stage tests whether the segments actually differ in ion intensity, and by
how much.

> This notebook is a thin wrapper around the `src/` package. Every computation
> below is the same function the CLI calls, so the notebook and
> `python scripts/run_pipeline.py` produce identical results. To change a
> parameter, edit `configs/pipeline_config.yaml` rather than the code here.

In [ ]:
import sys
from pathlib import Path

# Locate the project root so `src` imports work wherever Jupyter was started from.
ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
from src.config import load_config

config = load_config()
# Show paths relative to the project root so notebooks stay portable.
print("Project  :", ROOT.name)
print("Raw data :", config.raw_dir.relative_to(ROOT) if ROOT in config.raw_dir.parents else config.raw_dir)
print("Outputs  :", config.processed_dir.relative_to(ROOT) if ROOT in config.processed_dir.parents else config.processed_dir)

In [ ]:
import numpy as np

from src.features.selection import build_cube, select_analysis_channels
from src.preprocessing.stack_io import load_clean_stack
from src.segmentation import cluster
from src.viz.style import segment_names

segments = cluster.load_segments(config.processed_dir)
fg_idx = np.where(segments["foreground_mask"].astype(bool))[0]
labels = segments["labels_dominant"]

images, metadata = load_clean_stack(config.processed_dir)
selection = select_analysis_channels(images, metadata, config)
cube = build_cube(images, selection)
cube_fg = cube.reshape(-1, cube.shape[-1])[fg_idx]

names = segment_names(len(set(labels)), config)
print(f"{cube_fg.shape[0]:,} pixels, {cube_fg.shape[1]} channels, "
      f"{len(set(labels))} segments")

## Segment chemistry

In [ ]:
from src.stats import validation

profiles = validation.segment_chemistry_profiles(
    cube_fg, labels, selection.labels, names
)
profiles.round(2)

## Kruskal-Wallis per channel

Non-parametric, because ion-count distributions are heavily zero-inflated.
With tens of thousands of pixels almost anything reaches significance, so the
effect size is what carries the meaning.

In [ ]:
results, posthoc = validation.kruskal_dunn_by_channel(
    cube_fg, labels, selection.labels,
    p_adjust=config.get("stats.p_adjust", "bonferroni"),
)
results

## Which pairs of segments differ?

Dunn's post-hoc on the channel with the largest effect size.

In [ ]:
from src.viz import stats_plots

best = validation.strongest_effect_channel(results)
print(f"largest effect: {best}")

stats_plots.plot_dunn_heatmap(
    posthoc[best], best, config.figure_path("04_dunn_posthoc_heatmap.png")
)
posthoc[best].round(4)

## Cross-polarity colocalization

Does the secondary-mode ion concentrate where any segment sits? The secondary
acquisition is resampled onto the analysis grid first.

In [ ]:
from src.features.selection import resample_to

coloc = None
if selection.secondary_keys:
    secondary = resample_to(images[selection.secondary_keys[0]], selection.shape)
    secondary_flat = secondary.reshape(-1)[fg_idx]
    coloc = validation.colocalization(secondary_flat, labels, segment_names=names)
    stats_plots.plot_colocalization(
        coloc, config.figure_path("04_cross_polarity_coloc.png")
    )
coloc

In [ ]:
written = validation.save_statistics(config.processed_dir, results, posthoc, coloc)
for name, path in written.items():
    print(f"{name:16s} -> {path.name}")

Equivalent CLI command:

```bash
python scripts/run_pipeline.py --stage stats
```